In [ ]:
!nvidia-smi

Wed May 27 23:24:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

!pip -q install kaggle

os.environ['KAGGLE_USERNAME'] = 'habibak'
os.environ['KAGGLE_API_TOKEN'] = '242afd31d82feb6c9e7bc8a8549ddfdc'

!kaggle datasets download -d muhammetzahitaydn/hardhat-vest-dataset-v3

!unzip -nq hardhat-vest-dataset-v3.zip -d ppe_data

Dataset URL: https://www.kaggle.com/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3
License(s): CC0-1.0
100% 4.21G/4.21G [00:41<00:00, 109MB/s]



In [ ]:
!ls -l ppe_data

total 8
drwxr-xr-x 5 root root 4096 May 27 23:25 images
drwxr-xr-x 5 root root 4096 May 27 23:26 labels


In [ ]:
!find ppe_data -name "*.txt" | head -n 5

ppe_data/labels/train/000001.txt
ppe_data/labels/train/000002.txt
ppe_data/labels/train/000003.txt
ppe_data/labels/train/000004.txt
ppe_data/labels/train/000005.txt


In [ ]:
!head -n 15 ppe_data/labels/train/*.txt | head -n 60    # this shows we have 3 classes but we'll check the classes.txt

==> ppe_data/labels/train/000001.txt <==
0 0.408594 0.300000 0.095312 0.134375
0 0.239844 0.421875 0.048438 0.065625

==> ppe_data/labels/train/000002.txt <==
0 0.603125 0.256250 0.093750 0.150000
0 0.320312 0.327344 0.078125 0.126562
1 0.668750 0.426563 0.231250 0.221875

==> ppe_data/labels/train/000003.txt <==
0 0.8587499808054417 0.09469697251915932 0.1424999968148768 0.1666666716337204
0 0.4737499894108623 0.6458333525806665 0.07249999837949872 0.1401515193283558
0 0.3474999922327697 0.10227273032069206 0.08999999798834324 0.14393939822912216
0 0.6387499857228249 0.2367424312978983 0.06749999849125743 0.1250000037252903

==> ppe_data/labels/train/000004.txt <==
2 0.682813 0.320312 0.081250 0.168750
2 0.771875 0.300781 0.121875 0.229687
2 0.921094 0.379688 0.110937 0.190625
2 0.628125 0.341406 0.065625 0.126562
2 0.582031 0.370312 0.057813 0.103125

==> ppe_data/labels/train/000005.txt <==
0 0.176563 0.385156 0.200000 0.323437
0 0.716406 0.453906 0.151562 0.251563

==> ppe_data/lab

###These files show us only 3 classes but if we look at our classes.txt file it has 4 classes so we will read that instead

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  data = {
      'path': '/content/ppe_data',
      'train': 'images/train',
      'val': 'images/val',
      'test': 'images/test',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')
  return

path_to_classes_txt = '/content/ppe_data/labels/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

Created config file at /content/data.yaml

File contents:

path: /content/ppe_data
train: images/train
val: images/val
test: images/test
nc: 4
names:
- helmet
- vest
- head
- person


# Now let's train our model!


In [ ]:
!pip install ultralytics        # needed for YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive         # save our results
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
 !yolo detect train data=/content/data.yaml model=yolov8s.pt epochs=80 imgsz=640 cache=False project=/content/drive/MyDrive/YOLOv8_PPE_Detection

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, i

In [ ]:
'''
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/YOLO_PPE/train-2/weights/last.pt')
model.train(resume=True)
'''

"\nfrom ultralytics import YOLO\n\nmodel = YOLO('/content/drive/MyDrive/YOLO_PPE/train-2/weights/last.pt')\nmodel.train(resume=True)\n"

# Now, Let's Do Some Model Evaluation and Data Auditing to Make Sure Everything is Going Well

### Note: I started a new runtime for this part after saving my results hence why I reinstalled and reimported some stuff alongside remounting my google drive.

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/YOLOv8_PPE_Detection/train/weights/best.pt')     # load the best trained model weights

print("\nEverything is ready and our best model weights were loaded successfully.")
print("Classes detected by this model:", model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

Everything is ready and our best model weights were loaded successfully.
Classes detected by this model: {0: 'helmet', 1: 'vest', 2: 'head', 3: 'person'}


## We will do a visual and stats audit for our validation data

In [ ]:
# Validation: Statistics
print("📊 Calculating Validation Stats...")
val_results = model.val(data='data.yaml', split='val', conf=0.35, plots=True)

print("\n🎉 VALIDATION SET SCORES:")
print("=" * 60)
print(f"🏆 Val Precision:  {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"🏆 Val Recall:     {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"🏆 Val mAP50:      {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"🏆 Val mAP50-95:   {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print("=" * 60)

📊 Calculating Validation Stats...
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1829.0±1234.8 MB/s, size: 263.8 KB)
val: Scanning /content/ppe_data/labels/val... 2438 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2438/2438 1.3Kit/s 1.9s
val: New cache created: /content/ppe_data/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 153/153 14.8s/it 37:43
                   all       2438      21032      0.878      0.861      0.831      0.526
                helmet       1782       6586      0.932       0.89      0.886      0.601
                  vest        406        870      0.801      0.774      0.729      0.501
                  head        712      13576      0.901      0.919      0.878      0.478
Speed: 7.1ms preprocess, 909.3ms inferenc

In [ ]:
# Validation: Visual Audit
print("🖼️ Generating Validation Prediction Images...")
model.predict(
    source='ppe_data/images/val',
    batch=16,
    conf=0.35,
    save=True,
    plots=True,
    name='val_audit_images',
    exist_ok=True
)

🖼️ Generating Validation Prediction Images...

image 1/2438 /content/ppe_data/images/val/000006.jpg: 640x640 5 heads, 830.5ms
image 2/2438 /content/ppe_data/images/val/000011.jpg: 640x640 19 helmets, 1 head, 830.5ms
image 3/2438 /content/ppe_data/images/val/000013.jpg: 640x640 1 helmet, 830.5ms
image 4/2438 /content/ppe_data/images/val/000024.jpg: 640x640 5 helmets, 5 heads, 830.5ms
image 5/2438 /content/ppe_data/images/val/000058.jpg: 640x640 4 helmets, 2 vests, 830.5ms
image 6/2438 /content/ppe_data/images/val/000068.jpg: 640x640 4 helmets, 830.5ms
image 7/2438 /content/ppe_data/images/val/000080.jpg: 640x640 7 helmets, 830.5ms
image 8/2438 /content/ppe_data/images/val/000083.jpg: 640x640 2 helmets, 830.5ms
image 9/2438 /content/ppe_data/images/val/000084.jpg: 640x640 8 helmets, 830.5ms
image 10/2438 /content/ppe_data/images/val/000092.jpg: 640x640 2 helmets, 830.5ms
image 11/2438 /content/ppe_data/images/val/000093.jpg: 640x640 1 helmet, 1 vest, 1 head, 830.5ms
image 12/2438 /conten

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'helmet', 1: 'vest', 2: 'head', 3: 'person'}
 obb: None
 orig_img: array([[[171, 184, 168],
         [171, 184, 168],
         [171, 184, 168],
         ...,
         [ 83, 124, 146],
         [ 83, 124, 146],
         [ 83, 124, 146]],
 
        [[167, 180, 164],
         [167, 180, 164],
         [167, 180, 164],
         ...,
         [ 83, 124, 146],
         [ 83, 124, 146],
         [ 83, 124, 146]],
 
        [[163, 176, 160],
         [163, 176, 160],
         [164, 177, 161],
         ...,
         [ 83, 124, 146],
         [ 83, 124, 146],
         [ 83, 124, 146]],
 
        ...,
 
        [[  7,   9,  19],
         [  7,   9,  19],
         [  7,   9,  19],
         ...,
         [  1,   7,  26],
         [  1,   7,  26],
         [  1,   7,  26]],
 
        [[  7,   9,  19],
         [  7,   9,  19],
         [  7,   9,  19

## Now, we will do the same for our test data

In [ ]:
# Test: Statistics
print("📊 Calculating Final Test Stats...")
test_results = model.val(data='data.yaml', split='test', conf=0.35, plots=False)      # plots= false because we don't want to generate confusion matrix and so on for test data

print("\n🎉 FINAL TEST SCORES:")
print("=" * 60)
print(f"🏆 Final Test Precision:  {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"🏆 Final Test Recall:     {test_results.results_dict['metrics/recall(B)']:.4f}")
print(f"🏆 Final Test mAP50:      {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"🏆 Final Test mAP50-95:   {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print("=" * 60)

📊 Calculating Final Test Stats...
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 62.7±53.3 MB/s, size: 239.7 KB)
val: Scanning /content/ppe_data/labels/test... 2455 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2455/2455 386.0it/s 6.4s
val: New cache created: /content/ppe_data/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 154/154 14.4s/it 36:54
                   all       2455      20193      0.887       0.86      0.834      0.534
                helmet       1822       6749       0.94      0.875      0.867        0.6
                  vest        440        935      0.812      0.783      0.759      0.519
                  head        713      12509       0.91      0.922      0.877      0.483
Speed: 4.4ms preprocess, 884.6ms inference, 0.0ms loss, 0.9ms postprocess per image

🎉 FINAL TEST SCORES:
🏆 Final Test Preci

In [ ]:
# Test: Visual Audit
print("🖼️ Generating Final Test Prediction Images...")
model.predict(
    source='ppe_data/images/test',
    conf=0.35,
    save=True,
    name='test_audit_images',
    exist_ok=True
)

🖼️ Generating Final Test Prediction Images...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2455 /content/ppe_data/images/test/000009.jpg: 640x640 1 helmet, 651.4ms
image 2/2455 /content/ppe_data/images/test/000010.jpg: 640x640 5 helmets, 5 vests, 607.1ms
image 3/2455 /content/ppe_data/images/test/000021.jpg: 640x640 1 helmet, 2 vests, 583.3ms
image 4/2455 /content/ppe_data/images/test/000022.jpg: 640x640 7 helmets, 692.0ms
image 5/2455 /content/ppe_data/images/test/000034.jpg: 640x640 1 hel

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'helmet', 1: 'vest', 2: 'head', 3: 'person'}
 obb: None
 orig_img: array([[[ 24,  29,  28],
         [ 23,  28,  27],
         [ 23,  28,  27],
         ...,
         [ 29,  31,  32],
         [ 29,  31,  32],
         [ 27,  29,  30]],
 
        [[ 24,  29,  28],
         [ 24,  29,  28],
         [ 23,  28,  27],
         ...,
         [ 28,  30,  31],
         [ 28,  30,  31],
         [ 26,  28,  29]],
 
        [[ 24,  29,  28],
         [ 24,  29,  28],
         [ 23,  28,  27],
         ...,
         [ 31,  33,  34],
         [ 30,  32,  33],
         [ 30,  32,  33]],
 
        ...,
 
        [[ 13,  17,  18],
         [ 13,  17,  18],
         [ 13,  17,  18],
         ...,
         [134, 131, 117],
         [133, 130, 115],
         [130, 127, 112]],
 
        [[ 13,  17,  18],
         [ 13,  17,  18],
         [ 13,  17,  18

## We will now save our final audit results to our drive

In [ ]:
!zip -r /content/drive/MyDrive/YOLOv8_PPE_Detection/PPE_Detection_Model_Evaluation.zip runs/
print("Final results are backed up.")

  adding: runs/ (stored 0%)
  adding: runs/detect/ (stored 0%)
  adding: runs/detect/val/ (stored 0%)
  adding: runs/detect/val/val_batch1_labels.jpg (deflated 5%)
  adding: runs/detect/val/BoxR_curve.png (deflated 12%)
  adding: runs/detect/val/BoxF1_curve.png (deflated 12%)
  adding: runs/detect/val/val_batch2_labels.jpg (deflated 7%)
  adding: runs/detect/val/BoxPR_curve.png (deflated 17%)
  adding: runs/detect/val/confusion_matrix_normalized.png (deflated 27%)
  adding: runs/detect/val/val_batch0_pred.jpg (deflated 7%)
  adding: runs/detect/val/BoxP_curve.png (deflated 18%)
  adding: runs/detect/val/val_batch2_pred.jpg (deflated 6%)
  adding: runs/detect/val/confusion_matrix.png (deflated 25%)
  adding: runs/detect/val/val_batch0_labels.jpg (deflated 8%)
  adding: runs/detect/val/val_batch1_pred.jpg (deflated 5%)
  adding: runs/detect/val_audit_images/ (stored 0%)
  adding: runs/detect/val_audit_images/bpart2_001049.jpg (deflated 0%)
  adding: runs/detect/val_audit_images/006992.jp

# Now let's start our video tracking section (Auditing and Tracking along with saving our results)

### Note: do not run this cell if you already installed ultralytics in your current session I did this because I started a new session here.

In [ ]:
# install ultralytics for this session
!pip install -U ultralytics lapx -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

!yolo checks

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.2/235.7 GB disk)

OS                     Linux-6.6.122+-x86_64-with-glibc2.35
Environment            Colab
Python                 3.12.13
Install                pip
Path                   /usr/local/lib/python3.12/dist-packages/ultralytics
RAM                    12.67 GB
Disk                   47.2/235.7 GB
CPU                    Intel Xeon CPU @ 2.00GHz
CPU count              2
GPU                    Tesla T4, 14913MiB
GPU count              1
CUDA                   12.8

numpy                  ✅ 2.0.2>=1.23.0
matplotlib             ✅ 3.10.0>=3.3.0
opencv-p

In [ ]:
# sanity check
import ultralytics
import lap
print(f"Ultralytics: {ultralytics.__version__}")
print("Both packages installed and ready.")

Ultralytics: 8.4.60
Both packages installed and ready.


In [ ]:
from google.colab import drive
import os
import pandas as pd
from ultralytics import YOLO
import yaml

# we will mount our drive and prepare our folder paths here
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/'
videos_path = os.path.join(base_path, 'ppe_videos/')
eval_path = os.path.join(base_path, 'ppe_tracking_evaluation/')

# create folder for results/ evaluation
if not os.path.exists(eval_path):
    os.makedirs(eval_path)
    print(f"Created: {eval_path} successfully")

print("Everything is ready. Drive mounted and folders linked.")

Mounted at /content/drive
Created: /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_tracking_evaluation/ successfully
Everything is ready. Drive mounted and folders linked.


In [ ]:
# Video Auditing and Tracking here
# load our best weights model again
model = YOLO('/content/drive/MyDrive/YOLOv8_PPE_Detection/train/weights/best.pt')

# make it global so we don't have to change two values later
conf_threshold = 0.35

# loop through videos so we don't have to write each name and then store them in a list
videos = [f for f in os.listdir(videos_path) if f.endswith('.mp4')]

# list to store audit results
audit_data = []


#  Audit Videos (includes tracking and getting results)
for video_name in videos:
    print(f"\nStarting Audit: {video_name} ---")

    # Video Tracking
    try:
        results = model.track(source=os.path.join(videos_path, video_name),
                              save=True,
                              project=eval_path,
                              name=f"{video_name.replace('.mp4', '')}_conf_{conf_threshold}",       # have the threshold in the video name to differentiate between different confidence threshold's
                              conf=conf_threshold,       # start with the best one we reached in image auditing then we can adjust later
                              stream=True,
                              persist=True,
                              tracker="/content/drive/MyDrive/YOLOv8_PPE_Detection/my_tracker.yaml")    # buffer for more patience to reduce id flickers

        frame_count = 0
        total_detections = 0

        for r in results:
            frame_count += 1
            if r.boxes:
                    total_detections += len(r.boxes)

        avg_per_frame = total_detections / frame_count if frame_count > 0 else 0    # tells us the detection density or the average detections per frame

        print(f"Audit Complete for: {video_name}")
        print(f"Average Detections per Frame: {avg_per_frame:.4f}")

        audit_data.append({'Video Name': video_name, 'Avg Detections/Frame': avg_per_frame})    # in case the renaming fails so we can keep our audit results

        pd.DataFrame(audit_data).to_csv(os.path.join(eval_path,f"tracking_audit_summary_conf_{conf_threshold}.csv"),
                                                     index=False)

        print(f"✅ Audit data saved for: {video_name}")

        import time     # give it time to save
        time.sleep(20)

        run_folder = f"{video_name.replace('.mp4', '')}_conf_{conf_threshold}"
        run_dest = os.path.join(eval_path, run_folder)

        # find any video file YOLO actually saved
        video_files = [
            f for f in os.listdir(run_dest)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]

        if not video_files:
            raise FileNotFoundError(f"No output video found in {run_dest}")

        found_path = os.path.join(run_dest, video_files[0])

        # keep original extension
        original_ext = os.path.splitext(found_path)[1]

        new_filename = (
            f"{video_name.replace('.mp4', '')}"
            f"_conf_{conf_threshold}"
            f"{original_ext}"
        )

        output_path = os.path.join(run_dest, new_filename)

        if found_path != output_path:                        # see if file name differs and if so, rename it.
            if not os.path.exists(output_path):
                os.rename(found_path, output_path)
                print(f"✅ File renamed to: {new_filename}")
            else:
                print(f"ℹ️ File already exists: {new_filename}")
        else:
            print(f"ℹ️ File already has correct name: {new_filename}")


        print(f"✅ SUCCESS: {video_name} audit complete and file verified.")     # file renamed, verified and audit was successful


    except Exception as e:                 # final except if something goes wrong like a file not found error or memory error
        print(f"❌ An error occurred for: {video_name} !!!")       # this causes an unknown issue with YOLO paths but it still finds the file, tracks, then saves it correctly
        print(f"Reason: {e}")

# Final Summary (print it then save as csv later)
print("\n" + "="*40)
print("Tracking audit summary table")
print("="*40)

df = pd.DataFrame(audit_data)
print(df.to_string(index=False))

# save summary as csv (audit results are also saved per video in the loop above in case of crashes or errors)
summary_filename = f'tracking_audit_summary_conf_{conf_threshold}.csv'    # dynamic name per confidence threshold so we can easily optimize/compare results
df.to_csv(os.path.join(eval_path, summary_filename), index=False)
print(f"\nSummary saved to {eval_path}")

Streaming output truncated to the last 5000 lines.
video 1/1 (frame 122/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 7 helmets, 5 vests, 14.5ms
video 1/1 (frame 123/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 7 helmets, 5 vests, 19.4ms
video 1/1 (frame 124/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 6 helmets, 5 vests, 12.7ms
video 1/1 (frame 125/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 5 helmets, 4 vests, 18.3ms
video 1/1 (frame 126/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 6 helmets, 4 vests, 16.1ms
video 1/1 (frame 127/667) /content/drive/MyDrive/YOLOv8_PPE_Detection/ppe_tracking/ppe_videos/multiple_men_with_movements.mp4: 384x640 5 he

# THIS IS THE TRACKING SUMMARY BEFORE ADDING A FRAME BUFFER AND SLEEP SECONDS AND USING BOTSORT (first attempt)

========================================
Tracking audit summary table
========================================
                              Video Name  Avg Detections/Frame
                           from back.mp4              3.000000
                 walking one no vest.mp4              2.680804
                             welding.mp4              3.042636
                            360 view.mp4              4.089239
  multiple workers beside each other.mp4              4.079755
               multiple workers more.mp4              9.263889
                            cleaning.mp4              2.000000
         Construction-Site-CCTV.html.mp4              1.619880
                360 multiple workers.mp4              2.611111
                         two workers.mp4              2.729306
multiple men with movement and zooms.mp4              9.431784
         multi subject tracking away.mp4              6.300272
               two workers from side.mp4              1.867491
                  two workers stones.mp4              3.738095



## We made a custom tracker.yaml so we can use botsort and adjust things like frame buffer (we used 300 which means it waits 10 second for a 30 fps video for example.) and also things like track_high_thresh and track_high_thresh, which help with ID flickering and assigning an old ID to a new person because it thinks it looks like another one due to things like occulsion and the like. Using with_reid also helps with the recognition of a certain person's features so it doesn't confuse it with a similar worker since their attires are all really similar. Furthermore, using the Kalman Filter helps with the prediction of a person's position in the next frame, which is really helpful if they stop, turn, or are hidden behind an object (like a metal beam) or another worker. This all helps with the reduction of ID flickering and helps with better recognition of a worker's individual features which reduces the mixing up of IDs and improves tracking overall.